# LeetCode 397 — Integer Replacement bằng Python

Link đề: https://leetcode.com/problems/integer-replacement/ · Độ khó: **Medium** · Chủ đề: Greedy, Bit Manipulation, Memoization, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def integerReplacement(self, n: int) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; mọi ví dụ và đáp án trong notebook đều được chạy lại và đối chiếu bằng code.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên dương `n`; nếu `n` chẵn thì thay bằng `n / 2`, nếu `n` lẻ thì thay bằng `n + 1` **hoặc** `n - 1`; tìm **số phép biến đổi ít nhất** để `n` thành `1`.
2. Nắm công cụ Python: `%` và `//`, phép bit `n & 1`, `n >> 1`, `n & 3`, `bin(n)`, đệ quy, `functools.lru_cache`, hàng đợi `collections.deque` cho BFS.
3. Đi từ **đệ quy thử cả hai nhánh** → **đệ quy có nhớ (memoization)** → **BFS tìm đường ngắn nhất** → **tham lam theo hai bit cuối `O(log n)`**.
4. Hiểu mấu chốt tham lam: khi `n` lẻ, nếu hai bit cuối là `11` (và `n != 3`) thì **cộng 1** (cộng 1 làm "cuốn" cả dãy bit 1 liên tiếp, rút ngắn đường đi), ngược lại **trừ 1**; `n = 3` là ngoại lệ.
5. Tránh các lỗi kinh điển: luôn `n - 1` hoặc luôn `n + 1` khi `n` lẻ, quên ngoại lệ `n = 3`, tràn số ở ngôn ngữ 32-bit (`n + 1` với `n = 2^31 - 1`), đệ quy không nhớ bị lặp trạng thái.

## 2. Đề bài & ví dụ

Cho một số nguyên dương `n`, bạn có thể thực hiện các phép sau:

1. Nếu `n` **chẵn**, thay `n` bằng `n / 2`.
2. Nếu `n` **lẻ**, thay `n` bằng `n + 1` **hoặc** `n - 1`.

Trả về **số phép biến đổi tối thiểu** để `n` trở thành `1`.

**Ví dụ 1:** `n = 8` → `3` (`8 → 4 → 2 → 1`).

**Ví dụ 2:** `n = 7` → `4` (`7 → 8 → 4 → 2 → 1` hoặc `7 → 6 → 3 → 2 → 1`).

**Ví dụ 3:** `n = 4` → `2` (`4 → 2 → 1`).

**Ràng buộc:** `1 <= n <= 2^31 - 1`.

In [1]:
from collections import deque
from functools import lru_cache
import random
import time
from typing import List

examples = [
    (8, 3),
    (7, 4),
    (4, 2),
]
for n, expected in examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 8 -> kỳ vọng 3
n = 7 -> kỳ vọng 4
n = 4 -> kỳ vọng 2


### 2.1. Đọc kỹ đề

- Chẵn thì **bắt buộc** chia 2 (không có lựa chọn). Chỉ khi **lẻ** ta mới có hai lựa chọn: `n + 1` hay `n - 1`. Cả hai đều cho số **chẵn**, nên bước kế tiếp luôn là chia 2.
- Vì `n` đến `2^31 - 1`, ta không thể duyệt từng giá trị `1..n`; cần thuật toán cỡ `O(log n)`.
- Đáp án luôn tồn tại (từ số lẻ ≥ 3 ta luôn đi tới số chẵn rồi chia dần xuống). Hơn nữa, đáp án không vượt quá khoảng `2 * log2(n)`: mỗi lần chia 2 giảm một bit, mỗi bit tối đa kèm thêm một phép `±1`.

## 3. Kiến thức nền

### 3.1. Chẵn/lẻ bằng `%` và bằng bit

`n % 2 == 0` nghĩa là chẵn. Cách "bit": `n & 1` lấy **bit cuối cùng** (`0` nếu chẵn, `1` nếu lẻ). Phép chia 2 tương ứng **dịch phải một bit**: `n >> 1` (với số dương, bằng `n // 2`).

In [2]:
for n in [8, 7, 10, 13]:
    print(f"n={n:>2} bin={bin(n):>8} | n%2={n % 2} n&1={n & 1} | n//2={n // 2} n>>1={n >> 1}")

n= 8 bin=  0b1000 | n%2=0 n&1=0 | n//2=4 n>>1=4
n= 7 bin=   0b111 | n%2=1 n&1=1 | n//2=3 n>>1=3
n=10 bin=  0b1010 | n%2=0 n&1=0 | n//2=5 n>>1=5
n=13 bin=  0b1101 | n%2=1 n&1=1 | n//2=6 n>>1=6


### 3.2. `bin(n)` và hai bit cuối `n & 3`

`bin(n)` cho chuỗi nhị phân dạng `'0b111'`. Số `3` trong nhị phân là `11`, nên `n & 3` giữ lại **hai bit cuối** của `n` (một số từ `0` đến `3`). Với `n` lẻ: `n & 3 == 1` nghĩa là hai bit cuối là `01`, còn `n & 3 == 3` nghĩa là `11`. Phép `n % 4` cho cùng kết quả.

In [3]:
for n in [5, 7, 9, 11, 13, 15]:
    print(f"n={n:>2} {bin(n)[2:]:>5} | n&3={n & 3} (n%4={n % 4}) -> hai bit cuối {'11' if n & 3 == 3 else '01'}")

n= 5   101 | n&3=1 (n%4=1) -> hai bit cuối 01
n= 7   111 | n&3=3 (n%4=3) -> hai bit cuối 11
n= 9  1001 | n&3=1 (n%4=1) -> hai bit cuối 01
n=11  1011 | n&3=3 (n%4=3) -> hai bit cuối 11
n=13  1101 | n&3=1 (n%4=1) -> hai bit cuối 01
n=15  1111 | n&3=3 (n%4=3) -> hai bit cuối 11


### 3.3. Cộng 1 vào một dãy bit `1` liên tiếp

Trong nhị phân, `+1` **lan** qua mọi bit `1` ở đuôi: `0111 + 1 = 1000`. Một lần cộng làm biến mất ba bit `1` cùng lúc! Ngược lại `-1` trên số lẻ chỉ xóa đúng **một** bit `1` cuối: `0111 - 1 = 0110`. Đây là trực giác cho phần tham lam sau này.

In [4]:
for n in [7, 15, 23, 27]:
    print(f"{n:>2} = {bin(n)[2:]:>6}  | +1 -> {bin(n + 1)[2:]:>6}  | -1 -> {bin(n - 1)[2:]:>6}")

 7 =    111  | +1 ->   1000  | -1 ->    110
15 =   1111  | +1 ->  10000  | -1 ->   1110
23 =  10111  | +1 ->  11000  | -1 ->  10110
27 =  11011  | +1 ->  11100  | -1 ->  11010


### 3.4. Đệ quy và `functools.lru_cache`

Hàm gọi chính nó với bài toán nhỏ hơn gọi là **đệ quy**; cần **điều kiện dừng** (ở đây `n == 1`). `@lru_cache(None)` đặt trước hàm giúp Python **ghi nhớ** kết quả theo tham số — lần sau gọi lại với cùng tham số sẽ trả ngay, không tính lại (**memoization**).

In [5]:
calls = 0


def fib_plain(k):
    global calls
    calls += 1
    return k if k < 2 else fib_plain(k - 1) + fib_plain(k - 2)


calls_cached = 0


@lru_cache(None)
def fib_cached(k):
    global calls_cached
    calls_cached += 1
    return k if k < 2 else fib_cached(k - 1) + fib_cached(k - 2)


print(fib_plain(20), "số lần gọi không nhớ:", calls)
print(fib_cached(20), "số lần tính thật khi có nhớ:", calls_cached)

6765 số lần gọi không nhớ: 21891
6765 số lần tính thật khi có nhớ: 21


### 3.5. BFS bằng `deque`

**BFS** (duyệt theo tầng) tìm đường **ngắn nhất** trong đồ thị không trọng số: tầng 0 là điểm xuất phát, tầng `d` gồm các trạng thái cần đúng `d` bước. `deque` là hàng đợi hai đầu: `popleft()` lấy phần tử đầu rất nhanh (`O(1)`), khác với `list.pop(0)` là `O(n)`.

In [6]:
q = deque([10])
q.append(20)
q.append(30)
print(q.popleft(), q.popleft(), list(q))      # 10 20 [30]

10 20 [30]


## 4. Ý tưởng nền tảng: tính tay

Nhìn bài như **đồ thị**: mỗi số là một đỉnh, mỗi phép biến đổi là một cạnh (đều tốn 1 bước). Ta muốn **đường ngắn nhất** từ `n` về `1`.

Thử `n = 7`: lẻ nên có hai nhánh.

- Nhánh `7 → 8`: `8 → 4 → 2 → 1`, tổng `1 + 3 = 4` bước.
- Nhánh `7 → 6`: `6 → 3`, rồi `3` lẻ: `3 → 2 → 1`, tổng `1 + 1 + 2 = 4` bước.

Thử `n = 15`: `15 → 16 → 8 → 4 → 2 → 1` là `5` bước. Nhánh `15 → 14 → 7 → ...` thì `14 → 7` là hai bước và từ `7` cần thêm `4`, tổng `6` bước. Nhánh `+1` thắng.

Thử `n = 5`: `5 → 4 → 2 → 1` là `3` bước; nhánh `5 → 6 → 3 → 2 → 1` là `4` bước. Nhánh `-1` thắng.

Vậy lúc cộng, lúc trừ — cần một quy tắc. Trước tiên ta viết cách đệ quy làm đúng theo định nghĩa.

## 5. Cách 1 — Brute Force: đệ quy thử cả hai nhánh

**Ý tưởng:** gọi `f(n)` là số bước tối thiểu từ `n` về `1`.

- `f(1) = 0`.
- `n` chẵn: `f(n) = 1 + f(n / 2)`.
- `n` lẻ: `f(n) = 1 + min(f(n + 1), f(n - 1))`.

Đây là cách làm đúng theo đề, không cần nghĩ gì thêm.

In [7]:
def replace_recursive(n):
    if n == 1:
        return 0
    if n % 2 == 0:
        return 1 + replace_recursive(n // 2)
    return 1 + min(replace_recursive(n + 1), replace_recursive(n - 1))


for n, expected in examples:
    got = replace_recursive(n)
    print(n, "->", got, "| kỳ vọng", expected)
    assert got == expected

8 -> 3 | kỳ vọng 3
7 -> 4 | kỳ vọng 4
4 -> 2 | kỳ vọng 2


**Độ phức tạp:** mỗi lần gặp số lẻ ta rẽ hai nhánh nên cây gọi hàm phình ra. Đếm thử số lần gọi để thấy nút thắt.

In [8]:
calls = 0


def replace_counting(n):
    global calls
    calls += 1
    if n == 1:
        return 0
    if n % 2 == 0:
        return 1 + replace_counting(n // 2)
    return 1 + min(replace_counting(n + 1), replace_counting(n - 1))


print("Với n 'dễ' (ít cặp bit 01/11 xen kẽ):")
for n in [15, 255, 10**3, 10**6]:
    calls = 0
    ans = replace_counting(n)
    print(f"n={n:>8}: đáp án={ans:>2}, số lần gọi hàm={calls:>6}")

print("Với n xen kẽ bit 1010...101 (xấu nhất):")
for bits in [11, 17, 21]:
    n = int("10" * (bits // 2) + "1", 2)
    calls = 0
    ans = replace_counting(n)
    print(f"n={n:>8} ({bits} bit): đáp án={ans:>2}, số lần gọi hàm={calls:>6}")

Với n 'dễ' (ít cặp bit 01/11 xen kẽ):
n=      15: đáp án= 5, số lần gọi hàm=    19
n=     255: đáp án= 9, số lần gọi hàm=    57
n=    1000: đáp án=12, số lần gọi hàm=    70
n= 1000000: đáp án=24, số lần gọi hàm=  1174
Với n xen kẽ bit 1010...101 (xấu nhất):
n=    1365 (11 bit): đáp án=15, số lần gọi hàm=   716
n=   87381 (17 bit): đáp án=24, số lần gọi hàm= 12916
n= 1398101 (21 bit): đáp án=30, số lần gọi hàm= 88551


**Nút thắt:** với số "dễ" (như `10^6`) chỉ vài trăm lần gọi, nhưng với số có bit xen kẽ `1010...101` số lần gọi **nhân lên theo từng cặp bit** (kiểu Fibonacci): 11 bit tốn `716` lần, 17 bit tốn `12 916`, 21 bit tốn `88 551`; với `n = 1431655765` (31 bit, `1010...101`) ta đã đo ngoài notebook khoảng **10,9 triệu** lần gọi — quá chậm khi có nhiều test. Nguyên nhân: **trạng thái bị lặp**. Ví dụ từ `15`: nhánh `16 → 8` và nhánh `14 → 7 → 8` cùng đi qua `8`. Ta đang tính đi tính lại những số giống nhau. Điều này gợi ý **ghi nhớ** (Cách 2).

## 6. Ý tưởng thuật toán / bước đột phá

Bước 1: các trạng thái **không nhiều**. Từ `n`, sau mỗi lần chia 2, các số xuất hiện chỉ có dạng "xấp xỉ `n / 2^k`" (chỉ khoảng 2–3 giá trị khác nhau cho mỗi `k`). Vì thế tổng số trạng thái khác nhau chỉ cỡ `O(log n)`; **ghi nhớ** biến cây thành đường.

Bước 2: **nhìn nhị phân** để chọn `+1` hay `-1` ngay, không cần thử cả hai. Hãy in bảng "nhánh tốt nhất" cho các số lẻ nhỏ bằng quy hoạch động bottom-up (dùng làm sự thật để quan sát quy luật).

In [9]:
N = 64
dp = [0] * (N + 2)                       # dp[m] = số bước tối thiểu từ m về 1
for m in range(2, N + 2):
    if m % 2 == 0:
        dp[m] = dp[m // 2] + 1
    else:
        up = dp[(m + 1) // 2] + 2        # m -> m+1 (1 bước) rồi chia 2 (1 bước), tới (m+1)/2
        down = dp[m - 1] + 1             # m -> m-1 (1 bước), m-1 đã tính xong
        dp[m] = min(up, down)

print(" n | nhị phân | n&3 | dp[n] | dp[n+1] | dp[n-1] | nhánh tốt nhất")
for m in range(3, 40, 2):
    up, down = 1 + dp[m + 1], 1 + dp[m - 1]
    best = "+1" if up < down else ("-1" if down < up else "cả hai")
    print(f"{m:>2} | {bin(m)[2:]:>8} | {m & 3:>3} | {dp[m]:>5} | {dp[m + 1]:>7} | {dp[m - 1]:>7} | {best}")

 n | nhị phân | n&3 | dp[n] | dp[n+1] | dp[n-1] | nhánh tốt nhất
 3 |       11 |   3 |     2 |       2 |       1 | -1
 5 |      101 |   1 |     3 |       3 |       2 | -1
 7 |      111 |   3 |     4 |       3 |       3 | cả hai
 9 |     1001 |   1 |     4 |       4 |       3 | -1
11 |     1011 |   3 |     5 |       4 |       4 | cả hai
13 |     1101 |   1 |     5 |       5 |       4 | -1
15 |     1111 |   3 |     5 |       4 |       5 | +1
17 |    10001 |   1 |     5 |       5 |       4 | -1
19 |    10011 |   3 |     6 |       5 |       5 | cả hai
21 |    10101 |   1 |     6 |       6 |       5 | -1
23 |    10111 |   3 |     6 |       5 |       6 | +1
25 |    11001 |   1 |     6 |       6 |       5 | -1
27 |    11011 |   3 |     7 |       6 |       6 | cả hai
29 |    11101 |   1 |     7 |       6 |       6 | cả hai
31 |    11111 |   3 |     6 |       5 |       6 | +1
33 |   100001 |   1 |     6 |       6 |       5 | -1
35 |   100011 |   3 |     7 |       6 |       6 | cả hai
37 |   100

**Quy luật quan sát được:**

- Số lẻ có hai bit cuối `01` (`n & 3 == 1`): chọn **`-1`** (`5, 9, 13, ...`; có vài số hai nhánh hòa nhau như `29`, nhưng `-1` không bao giờ tệ hơn).
- Số lẻ có hai bit cuối `11` (`n & 3 == 3`): chọn **`+1`** (`15, 23, 31, ...`; ở `7, 11, 19` hai nhánh hòa nhau, `+1` không tệ hơn), ngoại trừ **`n = 3`**: nhìn dòng `n = 3` của bảng, `dp[4] = 2` và `dp[2] = 1`, nên `+1` tốn `1 + 2 = 3` bước còn `-1` tốn `1 + 1 = 2` bước: nhánh `-1` thắng.

**Vì sao?** `n` chia 2 dư 1; viết `n = 4m + 1` hoặc `n = 4m + 3`.

- `n = 4m + 1`: `-1` cho `4m`, đi tiếp hai lần chia 2 tới `m` hết `3` bước. `+1` cho `4m + 2 → 2m + 1` rồi vẫn còn lẻ, muốn tới mức `m` ít nhất cần thêm `2` bước nữa: tổng `4` bước. Vậy `-1` không tệ hơn.
- `n = 4m + 3`: `+1` cho `4m + 4 = 4(m+1)`, hai lần chia tới `m + 1` hết `3` bước, còn `-1` cho `4m + 2 → 2m + 1`, rồi `2m + 1` lẻ... đến mức `m + 1` cũng cần `≥ 4` bước. Vậy `+1` không tệ hơn (trừ trường hợp `m = 0` tức `n = 3`, vì mức "`m + 1 = 1`" quá thấp: `3 → 2 → 1` chỉ `2` bước).

(Đây là lập luận phác thảo; ta sẽ **kiểm chứng bằng máy** với mọi `n` đến vài chục nghìn ở mục 12.)

Ta có luật tham lam `O(log n)`.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (memoization)

**Giải thích từng đoạn:**

1. Giữ nguyên định nghĩa `f(n)` như Cách 1.
2. Thêm bộ nhớ (`dict` hoặc `lru_cache`) để mỗi `n` chỉ tính một lần.
3. Số trạng thái khác nhau chỉ `O(log n)`, nên tổng thời gian `O(log n)`.

Đếm số trạng thái thật sự được tính để thấy lợi ích.

In [10]:
def replace_memo(n):
    memo = {1: 0}

    def f(x):
        if x in memo:
            return memo[x]
        if x % 2 == 0:
            res = 1 + f(x // 2)
        else:
            res = 1 + min(f(x + 1), f(x - 1))
        memo[x] = res
        return res

    ans = f(n)
    return ans, len(memo)


for n in [15, 10**3, 10**6, 2**31 - 1]:
    ans, states = replace_memo(n)
    print(f"n={n:>10}: đáp án={ans:>2}, số trạng thái đã nhớ={states}")

n=        15: đáp án= 5, số trạng thái đã nhớ=10
n=      1000: đáp án=12, số trạng thái đã nhớ=22
n=   1000000: đáp án=24, số trạng thái đã nhớ=46
n=2147483647: đáp án=32, số trạng thái đã nhớ=91


In [11]:
# Code nộp LeetCode — Cách 2 (O(log n) time, O(log n) space)
class SolutionMemo:
    def integerReplacement(self, n: int) -> int:
        memo = {1: 0}

        def f(x: int) -> int:
            if x in memo:
                return memo[x]
            if x % 2 == 0:
                res = 1 + f(x // 2)
            else:
                res = 1 + min(f(x + 1), f(x - 1))
            memo[x] = res
            return res

        return f(n)


print([SolutionMemo().integerReplacement(n) for n, _ in examples])

[3, 4, 2]


**Độ phức tạp Cách 2:** Time `O(log n)` (số trạng thái), Space `O(log n)` (bộ nhớ + độ sâu đệ quy cỡ `2 * log2 n`).

### Cách 3 — BFS tìm đường ngắn nhất

**Giải thích từng đoạn:**

1. Đồ thị: đỉnh là số nguyên, cạnh theo luật đề (chẵn → một cạnh `x/2`, lẻ → hai cạnh `x+1`, `x-1`).
2. BFS từ `n`; dùng `seen` để không thăm lại. Tầng đầu tiên chứa `1` cho biết số bước tối thiểu.
3. Khác Cách 2 ở chỗ **không đệ quy**, dễ chứng minh "ngắn nhất" (BFS theo tầng).

In [12]:
def replace_bfs(n):
    if n == 1:
        return 0
    seen = {n}
    queue = deque([n])
    steps = 0
    while queue:
        for _ in range(len(queue)):                 # xử lý hết một tầng
            x = queue.popleft()
            if x == 1:
                return steps
            nxt = [x // 2] if x % 2 == 0 else [x + 1, x - 1]
            for y in nxt:
                if y not in seen:
                    seen.add(y)
                    queue.append(y)
        steps += 1
    return -1


print([replace_bfs(n) for n, _ in examples], replace_bfs(2**31 - 1))

[3, 4, 2] 32


In [13]:
# Code nộp LeetCode — Cách 3 (BFS)
class SolutionBFS:
    def integerReplacement(self, n: int) -> int:
        if n == 1:
            return 0
        seen = {n}
        queue = deque([n])
        steps = 0
        while queue:
            for _ in range(len(queue)):
                x = queue.popleft()
                if x == 1:
                    return steps
                for y in ([x // 2] if x % 2 == 0 else [x + 1, x - 1]):
                    if y not in seen:
                        seen.add(y)
                        queue.append(y)
            steps += 1
        return -1


print([SolutionBFS().integerReplacement(n) for n, _ in examples])

[3, 4, 2]


**Độ phức tạp Cách 3:** Time `O(log n)` (số đỉnh được thăm nhỏ, xem Cách 2), Space `O(log n)`.

### Cách 4 — Tham lam theo bit (chốt)

**Giải thích từng đoạn:**

1. Lặp tới khi `n == 1`.
2. Chẵn: `n >>= 1`.
3. Lẻ: nếu `n != 3` và `n & 3 == 3` thì `n += 1`, ngược lại `n -= 1`.
4. Mỗi vòng đếm một bước.

Mỗi hai bước `n` giảm ít nhất một nửa nên thời gian `O(log n)`.

In [14]:
def replace_greedy(n):
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n //= 2
        elif n != 3 and n % 4 == 3:
            n += 1
        else:
            n -= 1
        steps += 1
    return steps


print([replace_greedy(n) for n, _ in examples], replace_greedy(2**31 - 1))

[3, 4, 2] 32


In [15]:
# Code nộp LeetCode — Cách 4 (O(log n) time, O(1) space)
class SolutionGreedy:
    def integerReplacement(self, n: int) -> int:
        ans = 0
        while n != 1:
            if (n & 1) == 0:
                n >>= 1
            elif n != 3 and (n & 3) == 3:
                n += 1
            else:
                n -= 1
            ans += 1
        return ans


print([SolutionGreedy().integerReplacement(n) for n, _ in examples])

[3, 4, 2]


**Độ phức tạp Cách 4:** Time `O(log n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace tham lam với `n = 7`:

| bước | `n` (thập phân) | `n` nhị phân | `n & 3` | quyết định | `n` mới |
|---|---|---|---|---|---|
| 1 | 7 | `111` | 3 | lẻ, `&3==3`, `n != 3` → `+1` | 8 |
| 2 | 8 | `1000` | 0 | chẵn → `>> 1` | 4 |
| 3 | 4 | `100` | 0 | chẵn → `>> 1` | 2 |
| 4 | 2 | `10` | 2 | chẵn → `>> 1` | 1 |

Tổng `4` bước. Hàm `trace_greedy` in đúng các cột đó. Ngoài ra `trace_memo` in cây gọi của đệ quy có nhớ để thấy chỗ "trúng cache".

In [16]:
def trace_greedy(n):
    print(f"n gốc = {n} ({bin(n)[2:]})")
    print(f"{'bước':>4} | {'n':>6} | {'nhị phân':>12} | {'n&3':>3} | quyết định")
    steps = 0
    while n != 1:
        before = n
        if n % 2 == 0:
            n >>= 1
            why = "chẵn -> chia 2"
        elif n != 3 and n & 3 == 3:
            n += 1
            why = "lẻ, hai bit cuối 11 (n != 3) -> +1"
        else:
            n -= 1
            why = "lẻ, hai bit cuối 01 hoặc n == 3 -> -1"
        steps += 1
        print(f"{steps:>4} | {before:>6} | {bin(before)[2:]:>12} | {before & 3:>3} | {why} => {n}")
    print("tổng số bước:", steps)
    return steps


trace_greedy(7)

n gốc = 7 (111)
bước |      n |     nhị phân | n&3 | quyết định
   1 |      7 |          111 |   3 | lẻ, hai bit cuối 11 (n != 3) -> +1 => 8
   2 |      8 |         1000 |   0 | chẵn -> chia 2 => 4
   3 |      4 |          100 |   0 | chẵn -> chia 2 => 2
   4 |      2 |           10 |   2 | chẵn -> chia 2 => 1
tổng số bước: 4


4

In [17]:
trace_greedy(15)
print()
trace_greedy(3)            # ngoại lệ n = 3: dùng -1
print()
trace_greedy(59)           # 111011: trộn cả hai loại quyết định

n gốc = 15 (1111)
bước |      n |     nhị phân | n&3 | quyết định
   1 |     15 |         1111 |   3 | lẻ, hai bit cuối 11 (n != 3) -> +1 => 16
   2 |     16 |        10000 |   0 | chẵn -> chia 2 => 8
   3 |      8 |         1000 |   0 | chẵn -> chia 2 => 4
   4 |      4 |          100 |   0 | chẵn -> chia 2 => 2
   5 |      2 |           10 |   2 | chẵn -> chia 2 => 1
tổng số bước: 5

n gốc = 3 (11)
bước |      n |     nhị phân | n&3 | quyết định
   1 |      3 |           11 |   3 | lẻ, hai bit cuối 01 hoặc n == 3 -> -1 => 2
   2 |      2 |           10 |   2 | chẵn -> chia 2 => 1
tổng số bước: 2

n gốc = 59 (111011)
bước |      n |     nhị phân | n&3 | quyết định
   1 |     59 |       111011 |   3 | lẻ, hai bit cuối 11 (n != 3) -> +1 => 60
   2 |     60 |       111100 |   0 | chẵn -> chia 2 => 30
   3 |     30 |        11110 |   2 | chẵn -> chia 2 => 15
   4 |     15 |         1111 |   3 | lẻ, hai bit cuối 11 (n != 3) -> +1 => 16
   5 |     16 |        10000 |   0 | chẵn -> chia 2 =>

8

In [18]:
def trace_memo(n):
    memo = {1: 0}
    log = []

    def f(x, depth):
        pad = "  " * depth
        if x in memo:
            log.append(f"{pad}f({x}) = {memo[x]}   <- lấy từ bộ nhớ")
            return memo[x]
        if x % 2 == 0:
            log.append(f"{pad}f({x}): chẵn -> 1 + f({x // 2})")
            res = 1 + f(x // 2, depth + 1)
        else:
            log.append(f"{pad}f({x}): lẻ -> 1 + min(f({x + 1}), f({x - 1}))")
            res = 1 + min(f(x + 1, depth + 1), f(x - 1, depth + 1))
        memo[x] = res
        log.append(f"{pad}=> f({x}) = {res}")
        return res

    ans = f(n, 0)
    print("\n".join(log))
    return ans


print("đáp án:", trace_memo(15))

f(15): lẻ -> 1 + min(f(16), f(14))
  f(16): chẵn -> 1 + f(8)
    f(8): chẵn -> 1 + f(4)
      f(4): chẵn -> 1 + f(2)
        f(2): chẵn -> 1 + f(1)
          f(1) = 0   <- lấy từ bộ nhớ
        => f(2) = 1
      => f(4) = 2
    => f(8) = 3
  => f(16) = 4
  f(14): chẵn -> 1 + f(7)
    f(7): lẻ -> 1 + min(f(8), f(6))
      f(8) = 3   <- lấy từ bộ nhớ
      f(6): chẵn -> 1 + f(3)
        f(3): lẻ -> 1 + min(f(4), f(2))
          f(4) = 2   <- lấy từ bộ nhớ
          f(2) = 1   <- lấy từ bộ nhớ
        => f(3) = 2
      => f(6) = 3
    => f(7) = 4
  => f(14) = 5
=> f(15) = 5
đáp án: 5


## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 4** (tham lam theo bit): `O(log n)` thời gian, `O(1)` bộ nhớ.

In [19]:
class Solution:
    def integerReplacement(self, n: int) -> int:
        ans = 0
        while n != 1:
            if (n & 1) == 0:             # chẵn: chia 2 bằng dịch bit
                n >>= 1
            elif n != 3 and (n & 3) == 3:  # lẻ, hai bit cuối 11, trừ n = 3: cộng 1
                n += 1
            else:                        # lẻ, hai bit cuối 01 (hoặc n = 3): trừ 1
                n -= 1
            ans += 1
        return ans


for n, expected in examples:
    got = Solution().integerReplacement(n)
    print(n, "->", got)
    assert got == expected

8 -> 3
7 -> 4
4 -> 2


## 10. Vì sao đúng

1. **Chẵn bắt buộc chia 2:** đề không cho lựa chọn khác, nên mọi cách giải đều phải chia.
2. **Lẻ có hai nhánh, đều dẫn tới số chẵn rồi chia 2.** Nên so sánh hai nhánh chính là so sánh `f((n+1)/2)` và `f((n-1)/2)` (hai số kề nhau).
3. **Chứng minh thực nghiệm kết hợp phác thảo toán:** phân tích `n = 4m + 1` (chọn `-1`) và `n = 4m + 3` (chọn `+1` trừ `n = 3`) như mục 6; ta còn kiểm chứng **mọi `n` từ `1` đến `20000`** với quy hoạch động bottom-up và BFS ở mục 12 — hai oracle độc lập với luật tham lam.
4. **Kết thúc:** mỗi vòng, `n` chẵn thì giảm một nửa; `n` lẻ thì sau `±1` thành chẵn và vòng sau giảm một nửa, nên sau tối đa `2 * 31` vòng thì tới `1`. Không có chu kỳ vì `n = 3` bị buộc `-1` (không thể `3 → 4 → 2 → 3`...).
5. **Không tràn số trong Python;** ở Java/C++ phải xử lý `n = 2^31 - 1` (xem lỗi 4 ở mục 13).

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy thuần | tăng theo cấp số nhân của số bit (tệ nhất cỡ `10^7` lần gọi với `n` 31 bit) | `O(log n)` |
| 2. Đệ quy có nhớ | `O(log n)` | `O(log n)` |
| 3. BFS | `O(log n)` | `O(log n)` |
| 4. Tham lam | **`O(log n)`** | **`O(1)`** |

Giải thích Cách 4: mỗi vòng lặp ứng với một thao tác; sau một lần `±1` (nếu có) là một lần chia 2 làm mất một bit; số bit của `n` tối đa `31` nên số vòng `<= 2 * 31 = 62`.

## 12. Bộ test

Dùng **hai oracle độc lập** với tham lam: (a) quy hoạch động bottom-up `dp` trên mọi `n` đến `N`; (b) BFS. Cùng với đệ quy có nhớ cho `n` lớn.

In [20]:
# 12.1 Ví dụ của đề + biên
cases = [(8, 3), (7, 4), (4, 2), (1, 0), (2, 1), (3, 2), (5, 3), (6, 3), (9, 4), (15, 5),
         (2**31 - 1, 32), (2**30, 30), (2**30 + 1, 31)]
solutions = [Solution(), SolutionMemo(), SolutionBFS(), SolutionGreedy()]
for n, expected in cases:
    for sol in solutions:
        got = sol.integerReplacement(n)
        assert got == expected, (type(sol).__name__, n, got, expected)
print("OK: ví dụ đề + biên cho cả 4 cách")

OK: ví dụ đề + biên cho cả 4 cách


In [21]:
# 12.2 Oracle bottom-up dp: đối chiếu MỌI n từ 1 đến N
N = 20000
dp = [0] * (N + 2)
for m in range(2, N + 2):
    if m % 2 == 0:
        dp[m] = dp[m // 2] + 1
    else:
        dp[m] = min(dp[(m + 1) // 2] + 2, dp[m - 1] + 1)

sol = Solution()
for n in range(1, N + 1):
    assert sol.integerReplacement(n) == dp[n], n
print(f"OK: tham lam khớp dp bottom-up với mọi n trong [1, {N}]")

OK: tham lam khớp dp bottom-up với mọi n trong [1, 20000]


In [22]:
# 12.3 Đối chiếu bốn cách + đệ quy thuần trên n nhỏ (1..400)
for n in range(1, 401):
    expect = dp[n]
    assert replace_recursive(n) == expect
    assert SolutionMemo().integerReplacement(n) == expect
    assert SolutionBFS().integerReplacement(n) == expect
    assert SolutionGreedy().integerReplacement(n) == expect
print("OK: đệ quy thuần = memo = BFS = tham lam trên n = 1..400")

OK: đệ quy thuần = memo = BFS = tham lam trên n = 1..400


In [23]:
# 12.4 Ngẫu nhiên có seed, n lớn đúng ràng buộc [1, 2^31 - 1]: tham lam vs memo vs BFS
random.seed(397)
for _ in range(3000):
    n = random.randint(1, 2**31 - 1)
    a = Solution().integerReplacement(n)
    b = SolutionMemo().integerReplacement(n)
    c = SolutionBFS().integerReplacement(n)
    assert a == b == c, (n, a, b, c)
print("OK: 3000 số ngẫu nhiên lớn, tham lam = memo = BFS")

OK: 3000 số ngẫu nhiên lớn, tham lam = memo = BFS


In [24]:
# 12.5 Các dạng "khó": toàn bit 1 (2^k - 1), 2^k + 1, 2^k - 3, 2^k + 3
for k in range(2, 31):
    for n in [2**k - 1, 2**k + 1, 2**k - 3, 2**k + 3]:
        if 1 <= n <= 2**31 - 1:
            assert Solution().integerReplacement(n) == SolutionMemo().integerReplacement(n), n
print("OK: các dạng sát lũy thừa 2")
print("2^31 - 1 ->", Solution().integerReplacement(2**31 - 1), "| 2^30 + 1 ->", Solution().integerReplacement(2**30 + 1))

OK: các dạng sát lũy thừa 2
2^31 - 1 -> 32 | 2^30 + 1 -> 31


In [25]:
# 12.6 Chặn trên: đáp án không vượt 2 * floor(log2 n) (khoảng 62 với n lớn nhất)
for n in list(range(1, 2000)) + [random.randint(1, 2**31 - 1) for _ in range(500)]:
    assert Solution().integerReplacement(n) <= 2 * n.bit_length()
print("OK: đáp án <= 2 * số bit của n")

OK: đáp án <= 2 * số bit của n


In [26]:
# 12.7 Hiệu năng tham lam: 10^5 lần gọi với n lớn
random.seed(4)
nums = [random.randint(1, 2**31 - 1) for _ in range(10**5)]
t = time.perf_counter()
total = sum(Solution().integerReplacement(x) for x in nums)
elapsed = time.perf_counter() - t
assert elapsed < 5.0
print("10^5 lần gọi xong dưới 5 giây:", elapsed < 5.0, "| tổng bước =", total)

10^5 lần gọi xong dưới 5 giây: True | tổng bước = 3910001


## 13. Các lỗi người mới hay gặp

### Lỗi 1: luôn trừ 1 khi `n` lẻ (hoặc luôn cộng 1)

Hai chiến lược "đơn giản" đều sai, vì có số cần `-1` và có số cần `+1`.

In [27]:
def wrong_always_minus(n):
    steps = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else n - 1          # SAI: không bao giờ +1
        steps += 1
    return steps


def wrong_always_plus(n):
    steps = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else n + 1          # SAI: không bao giờ -1 (và n = 3 thành 3 bước)
        steps += 1
    return steps


bad_minus = next(n for n in range(1, 100) if wrong_always_minus(n) != Solution().integerReplacement(n))
bad_plus = next(n for n in range(1, 100) if wrong_always_plus(n) != Solution().integerReplacement(n))
print("luôn -1 sai đầu tiên tại n =", bad_minus, ":", wrong_always_minus(bad_minus), "so với đúng", Solution().integerReplacement(bad_minus))
print("luôn +1 sai đầu tiên tại n =", bad_plus, ":", wrong_always_plus(bad_plus), "so với đúng", Solution().integerReplacement(bad_plus))
assert bad_minus == 15 and bad_plus == 3

luôn -1 sai đầu tiên tại n = 15 : 6 so với đúng 5
luôn +1 sai đầu tiên tại n = 3 : 3 so với đúng 2


**Vì sao sai:** với `7` cả hai nhánh bằng nhau (`4` bước) nên chiến lược luôn `-1` chưa lộ lỗi; nó sai đầu tiên ở `15`: `15 → 14 → 7 → 6 → 3 → 2 → 1` là `6` bước, trong khi `15 → 16 → 8 → 4 → 2 → 1` chỉ `5` bước. Luôn `-1` thua ở các số có nhiều bit `1` liên tiếp (`15`, `31`, ...). Luôn `+1` thua ngay ở `3` (`3 → 4 → 2 → 1` là `3` bước thay vì `2`). **Cách sửa:** dùng luật theo hai bit cuối.

### Lỗi 2: quên ngoại lệ `n = 3`

In [28]:
def wrong_no_special_three(n):
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n //= 2
        elif n % 4 == 3:                                  # SAI: n = 3 cũng bị +1
            n += 1
        else:
            n -= 1
        steps += 1
    return steps


for n in [3, 4, 7]:
    print(n, "sai:", wrong_no_special_three(n), "| đúng:", Solution().integerReplacement(n))
assert wrong_no_special_three(3) == 3 and Solution().integerReplacement(3) == 2

3 sai: 3 | đúng: 2
4 sai: 2 | đúng: 2
7 sai: 4 | đúng: 4


**Vì sao sai:** `3 = 11` có hai bit cuối `11` nên luật chung bảo `+1` ⇒ `3 → 4 → 2 → 1` là `3` bước, trong khi `3 → 2 → 1` chỉ `2` bước. Sai lệch lan sang mọi số đi qua `3` (ví dụ `n = 6`, `n = 12`, ...). **Cách sửa:** `elif n != 3 and n & 3 == 3`.

### Lỗi 3: nhầm thứ tự ưu tiên toán tử bit

In [29]:
n = 7
print(n & 3 == 3, "|", (n & 3) == 3)           # cùng True: trong Python & ưu tiên cao hơn ==
print(8 & 1 == 0, "|", (8 & 1) == 0)           # cùng True
# Mô phỏng cách C/C++ hiểu "8 & 1 == 0": == ưu tiên hơn & nên thành 8 & (1 == 0) = 8 & False
print("kiểu C:", 8 & (1 == 0), "(sai ý định)")

True | True
True | True
kiểu C: 0 (sai ý định)


**Vì sao dễ nhầm:** trong Python, `&` có độ ưu tiên **cao hơn** `==` nên `n & 1 == 0` thực ra là `(n & 1) == 0`. Nhưng ở **C/C++/Java**, `==` ưu tiên **cao hơn** `&`: `n & 1 == 0` thành `n & (1 == 0)` — sai. Thói quen tốt: luôn đóng ngoặc `(n & 1) == 0`, nhất là khi chuyển code giữa các ngôn ngữ.

### Lỗi 4: tràn số 32-bit với `n = 2^31 - 1`

Trong Java/C++ kiểu `int`, `n + 1` khi `n = 2147483647` tràn thành số âm. Python không tràn, nhưng nếu bạn chuyển sang ngôn ngữ khác phải dùng `long`, hoặc đổi luật: với `n = 2^31 - 1` trả thẳng `32`.

In [30]:
def to_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x


n = 2**31 - 1
print("n + 1 trong Python:", n + 1, "| nếu là int32:", to_int32(n + 1))
print("đáp án đúng cho 2^31 - 1:", Solution().integerReplacement(n))
assert to_int32(n + 1) < 0

n + 1 trong Python: 2147483648 | nếu là int32: -2147483648
đáp án đúng cho 2^31 - 1: 32


**Vì sao sai:** nếu `n + 1` tràn thành số âm, vòng lặp chạy sai hoặc vô hạn. **Cách sửa:** dùng `long`/`int64`, hoặc xử lý riêng `n = Integer.MAX_VALUE`.

### Lỗi 5: đệ quy thuần cho `n` lớn

Không ghi nhớ nên số lần gọi lớn như đã đếm ở mục 5; nên thêm `memo`/`lru_cache` hoặc dùng tham lam.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia 2 khi chẵn, có lựa chọn khi lẻ" ⇒ bài **nhị phân**. Hãy luôn viết `bin(n)` ra giấy.
- **Mẫu tham lam bit:** xem **hai bit cuối** (`n & 3`) để quyết định; `+1` khi muốn "cuốn" nhiều bit `1` liền nhau.
- **Khi chưa tìm ra luật tham lam,** viết **brute force có nhớ** rồi **in bảng** như mục 6 để quan sát luật — kỹ thuật "chạy thực nghiệm trước, chứng minh sau".
- **Cây gọi hàm phình to** là dấu hiệu cần **memoization**. Đếm số lần gọi để kiểm tra giả thuyết.
- **Oracle:** luôn có một cách chậm nhưng chắc chắn đúng (DP/BFS) để so sánh với cách nhanh.

## 15. Biến thể

1. **Truy vết đường đi:** in một dãy số `n → ... → 1` tối ưu.
2. **Đếm số cách tối ưu:** có bao nhiêu đường đi ngắn nhất (ví dụ `7` có `2` đường).
3. **Chỉ cho `n - 1` và `n / 2` (khi chẵn):** bài đơn giản hơn: đáp án `= (số bit - 1) + (số bit 1 - 1)`.
4. **Thêm phép "nhân đôi", "chia 3"...:** đổi đồ thị, BFS vẫn dùng được.

In [31]:
def optimal_path(n):
    """Một đường đi tối ưu theo luật tham lam, dạng danh sách các giá trị."""
    path = [n]
    while n != 1:
        if n % 2 == 0:
            n //= 2
        elif n != 3 and n % 4 == 3:
            n += 1
        else:
            n -= 1
        path.append(n)
    return path


def count_optimal_paths(n):
    ways = {}

    def best(x):
        if x == 1:
            return (0, 1)
        if x in ways:
            return ways[x]
        if x % 2 == 0:
            d, c = best(x // 2)
            res = (d + 1, c)
        else:
            d1, c1 = best(x + 1)
            d2, c2 = best(x - 1)
            if d1 < d2:
                res = (d1 + 1, c1)
            elif d2 < d1:
                res = (d2 + 1, c2)
            else:
                res = (d1 + 1, c1 + c2)
        ways[x] = res
        return res

    return best(n)


print(optimal_path(7))
print(optimal_path(15))
print(count_optimal_paths(7))              # (4, 2): 4 bước, 2 đường tối ưu
print(count_optimal_paths(15))

[7, 8, 4, 2, 1]
[15, 16, 8, 4, 2, 1]
(4, 2)
(5, 1)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | thử cả hai nhánh | rất lớn với `n` lớn | `O(log n)` | hiểu đề, `n` nhỏ |
| 2. Đệ quy có nhớ | bỏ trạng thái lặp | `O(log n)` | `O(log n)` | khi chưa thấy luật tham lam; **oracle** tốt |
| 3. BFS | đường ngắn nhất trên đồ thị | `O(log n)` | `O(log n)` | khi luật biến đổi phức tạp hơn đề gốc |
| 4. Tham lam bit | nhìn hai bit cuối | `O(log n)` | `O(1)` | **nộp bài**, phỏng vấn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bin(13)`; (b) `13 & 3`; (c) `Solution().integerReplacement(16)`; (d) `Solution().integerReplacement(2)`; (e) `5 >> 1`.

**Bài 2 (tính tay).** Với `n = 23` (nhị phân `10111`): viết từng bước tham lam (`n`, nhị phân, quyết định) và tổng số bước; rồi thử nhánh còn lại ở bước đầu để thấy nó không tốt hơn.

**Bài 3 (điền TODO).** Hoàn thành `replace_todo(n)` theo Cách 4.

In [32]:
def replace_todo(n):
    # TODO 1: khởi tạo steps = 0
    # TODO 2: while n != 1:
    #   TODO 3: nếu n chẵn thì chia 2
    #   TODO 4: nếu n lẻ, n != 3 và hai bit cuối là 11 thì n += 1, ngược lại n -= 1
    #   TODO 5: steps += 1
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `replace_dp(n)` bằng đệ quy có nhớ dùng `dict` (Cách 2) **không nhìn lại** mục 7; kiểm tra với `[8, 7, 4]` và `2**31 - 1`.

**Bài 5 (biến thể / mini challenge).** Viết `replacement_path(n)` trả về **một đường đi tối ưu** (danh sách các số từ `n` đến `1`) dùng BFS có truy vết cha, kèm hàm `is_valid_path(path, n)` kiểm tra: bắt đầu `n`, kết thúc `1`, từng bước đúng luật. Kiểm tra độ dài `path - 1` bằng đáp án.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'0b1101'`; `1`; `4`; `1`; `2`. Bài 3: nhìn lại hàm `replace_greedy` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `23 = 10111`, `n & 3 = 3`, nên `+1 → 24`; `24 → 12 → 6 → 3`; `3 → 2 → 1`. Tổng `1 + 3 + 2 = 6`. Bài 4: `memo = {1: 0}`; chẵn `1 + f(x // 2)`; lẻ `1 + min(f(x + 1), f(x - 1))`.
- **Mức 3 (sâu):** Bài 5: BFS giữ `parent = {n: None}`; khi pop được `1`, đi ngược `parent` để dựng đường, rồi `reverse`. `is_valid_path`: với mỗi cặp liên tiếp `(a, b)`, nếu `a` chẵn thì `b == a // 2`, nếu `a` lẻ thì `b in (a + 1, a - 1)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert bin(13) == "0b1101"
assert 13 & 3 == 1
assert Solution().integerReplacement(16) == 4
assert Solution().integerReplacement(2) == 1
assert 5 >> 1 == 2

# Bài 2: n = 23 -> 24 -> 12 -> 6 -> 3 -> 2 -> 1 (6 bước)
path23 = optimal_path(23)
assert path23 == [23, 24, 12, 6, 3, 2, 1]
assert len(path23) - 1 == 6 == Solution().integerReplacement(23)
# nhánh ngược ở bước đầu: 23 -> 22 -> 11 -> ... (không tốt hơn)
assert 1 + replace_recursive(22) >= 6


# Bài 3
def replace_todo_ans(n):
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n //= 2
        elif n != 3 and n & 3 == 3:
            n += 1
        else:
            n -= 1
        steps += 1
    return steps


for n, expected in examples:
    assert replace_todo_ans(n) == expected


# Bài 4
def replace_dp(n):
    memo = {1: 0}

    def f(x):
        if x not in memo:
            memo[x] = 1 + (f(x // 2) if x % 2 == 0 else min(f(x + 1), f(x - 1)))
        return memo[x]

    return f(n)


assert [replace_dp(n) for n in (8, 7, 4)] == [3, 4, 2]
assert replace_dp(2**31 - 1) == 32


# Bài 5
def replacement_path(n):
    parent = {n: None}
    queue = deque([n])
    while queue:
        x = queue.popleft()
        if x == 1:
            break
        for y in ([x // 2] if x % 2 == 0 else [x + 1, x - 1]):
            if y not in parent:
                parent[y] = x
                queue.append(y)
    path, cur = [], 1
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]


def is_valid_path(path, n):
    if path[0] != n or path[-1] != 1:
        return False
    for a, b in zip(path, path[1:]):
        if a % 2 == 0:
            if b != a // 2:
                return False
        elif b not in (a + 1, a - 1):
            return False
    return True


for n in list(range(1, 300)) + [2**31 - 1, 10**9]:
    p = replacement_path(n)
    assert is_valid_path(p, n)
    assert len(p) - 1 == Solution().integerReplacement(n)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (tham lam theo hai bit cuối)

```python
class Solution:
    def integerReplacement(self, n: int) -> int:
        ans = 0
        while n != 1:
            if (n & 1) == 0:
                n >>= 1
            elif n != 3 and (n & 3) == 3:
                n += 1
            else:
                n -= 1
            ans += 1
        return ans
```

**Checklist**

- [ ] Chẵn thì chia 2 (không có lựa chọn)?
- [ ] Lẻ: hai bit cuối `11` → `+1`, hai bit cuối `01` → `-1`?
- [ ] Đã xử lý ngoại lệ **`n = 3`** (dùng `-1`)?
- [ ] Đã đóng ngoặc `(n & 3) == 3` (an toàn khi chuyển sang C++/Java)?
- [ ] Với Java/C++: dùng `long` để `n + 1` không tràn khi `n = 2^31 - 1`?
- [ ] Đã kiểm chứng bằng DP/BFS với mọi `n` nhỏ và thử `1, 2, 3, 2^31 - 1`?